# 8.1 File Handling — Text

**Prerequisites:** 06 Exception Handling (context managers), 07 Module and Packages (pathlib)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What a file is, and what `open()` returns
- Modes: `r`, `w`, `a`, `x`, and the `+` variants
- **Why `with` is not optional**, and what happens without it
- 🔴 **Encoding** — and why omitting it is a portability bug
- Reading: `read`, `readline`, `readlines`, and iterating the file object
- `seek` and `tell`
- **`pathlib`** for one-line reads and writes
- **Atomic writes** — replacing a file without risking corruption
- Redirecting `print()` output safely

---

### What is a file?
- File is a named location on disk to store related information. 
- It is used to permanently store data in a non-volatile memory (e.g. hard disk).
    - Since, random access memory (RAM) is volatile which loses its data when computer is turned off, we use files for future use of the data.
- In data manipulation and analysis, you will often come across reading and writing files. 
- Python can handle files of different types including txt, JSON, HTML, CSV etc.
<br/><br/>
- When we want to read from or write to a file we need to open it first. 
    - When we are done, it needs to be closed, so that resources that are tied with the file are freed.
- Hence, in Python, a file operation takes place in the following order.
    - 1. Open a file
    - 2. Read or write (perform operation)
    - 3. Close the file

## Python file handling:
- **Coding step:**
```python
file = open('file_path', 'mode')  
data = file.read()   # if in read mode
file.write('any data') # if in write mode
file.close()
```
  
  
### Mode: 
<img src='./Image/8 Image a.png' width=85% height=40%>


### File Handling methods:
<img src='./Image/8 Image b.png' width=85% height=40%>

- **NOTE:** In python all escape sequences starts with backslash `\`.
    - Bydefault path copied from file in windows comes with `\` backslash and in Mac and Linux comes with `/` forwardslash.
    - Hence we must handle it carefully in windows by either using double backslash or by converting all backslash to forwardslash or using raw string 'r' with default copied path.

### Open a file:
- Python has a built-in function open() to open a file. 
- This function returns a file object, also called a handle, as it is used to read or modify the file accordingly.
- **Syntax:** open(file, mode='r', buffering=-1, encoding=None, errors=None, newline=None, closefd=True, opener=None)
    - The default is reading in text mode. In this mode, we get strings when reading from the file.

### Descripter:
- 'name', 'mode', 'closed' are data descripter to check properties of file.

---

### 🔴 Encoding: the argument you must not omit

`open()` has an `encoding` parameter. If you leave it out, Python uses a
**platform-dependent default** — historically `cp1252` on Windows and `UTF-8` on
macOS/Linux.

That is why a file written on one machine turns into mojibake, or raises
`UnicodeDecodeError`, on another. The code is identical; the default is not.

> ### The rule
> **Always pass `encoding="utf-8"` explicitly**, for both reading and writing.
> It costs nine characters and eliminates an entire class of bug.

> **Version note:** Python 3.10+ can warn about this — run with `-X warn_default_encoding`
> to find every unspecified `open()` in a codebase. From **3.15** the default is planned to
> become UTF-8 everywhere (PEP 686), but code that must run on older versions should stay
> explicit.

### The `errors=` parameter

When bytes cannot be decoded, you choose what happens:

| `errors=` | Behaviour |
|---|---|
| `"strict"` (default) | Raise `UnicodeDecodeError` — correct for data you control |
| `"replace"` | Substitute `\ufffd` (�) — good for logs and display |
| `"ignore"` | Drop the bad bytes — **silent data loss** |
| `"backslashreplace"` | Show `\xNN` escapes — useful for debugging |

### `utf-8` vs `utf-8-sig`

Excel and some Windows tools prepend a **BOM** (`EF BB BF`) to UTF-8 files. Read such a file
as `utf-8` and you get an invisible `\ufeff` at the start of the first field — a classic
"why doesn't my first column match?" bug. Read it as **`utf-8-sig`** and the BOM is stripped.

### A scratch directory for everything we write

Every example below that **writes** a file writes it into a throwaway directory from `tempfile`, not into the notes folder.

🔴 **This is a correction.** The original cells wrote straight into `File2Save/`. Three of them appended without ever truncating, so the files grew on every run and the notebook produced different output each time. One of the committed fixtures still shows the damage: `tab1.csv` contains `Neetu,negName,Corona Test`, which is an append with no trailing newline colliding with the next run's header.

`File2Save/` is still used, but only to **read** `Image.jpg`.

In [ ]:
import tempfile
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py08_text_"))
print("scratch directory:", WORK)

# msg2.txt is opened in 'r+' further down, which requires it to already
# exist, so seed it here. The committed fixture had 'This is line 6.' with
# no trailing newline, which is why line 8 ran into it.
(WORK / "msg2.txt").write_text(
    "This is line 5.\nThis is line 6.\n", encoding="utf-8"
)

FIXTURES = Path("File2Save")          # read-only: Image.jpg lives here
print("fixtures  :", FIXTURES.resolve())

In [ ]:
import locale, sys
from pathlib import Path

# ---- What does open() actually use when you do not say? ----
print("locale.getpreferredencoding():", locale.getpreferredencoding(False))
print("sys.getdefaultencoding()     :", sys.getdefaultencoding(), " <- for str, not files")

demo = WORK / "encoding_demo.txt"
text = "Grüße, नमस्कार, 日本語"

# ---- Always be explicit ----
demo.write_text(text, encoding="utf-8")
print("\nwritten as utf-8:", demo.stat().st_size, "bytes for", len(text), "characters")
print("read back correctly:", demo.read_text(encoding="utf-8") == text)

# ---- What goes wrong with the wrong encoding ----
raw = demo.read_bytes()
print("\nraw bytes  :", raw[:16])

try:
    wrong = raw.decode("ascii")
except UnicodeDecodeError as exc:
    print("as ascii   : UnicodeDecodeError at byte", exc.start)

mojibake = raw.decode("latin-1")
print("as latin-1 :", mojibake[:20], " <- mojibake, and NO error was raised")

# ---- errors= : what to do with undecodable bytes ----
print("\nerrors= strategies on the same bad decode:")
for strategy in ["replace", "ignore", "backslashreplace"]:
    out = raw.decode("ascii", errors=strategy)
    print(f"  {strategy:<18} {out[:26]!r}")

# ---- The UTF-8 BOM: what Excel and some Windows tools write ----
demo.write_text(text, encoding="utf-8-sig")           # adds a BOM
with_bom = demo.read_bytes()
print("\nutf-8-sig starts with BOM:", with_bom[:3] == b"\xef\xbb\xbf")
print("read with utf-8    :", repr(demo.read_text(encoding="utf-8")[:3]), " <- stray \\ufeff")
print("read with utf-8-sig:", repr(demo.read_text(encoding="utf-8-sig")[:3]), " <- BOM stripped")

demo.unlink()

### Write Mode:
- Open file in either `w` or `a` or `x` mode.
- To write data onto a file use `<file>.write()`. 
    - The write() method does not add a newline character `\n` to the end.

#### w mode:
- `w` or `wt` mode is used for writing on file or on text format file respectively.
- In both `w` and `wt` mode 
    - if file doesn't exist already then first it will create file then open it.
    - But if file exist then the new content will overwrite on the previously written data.

In [ ]:
file = open(WORK / "msg1.txt", 'wt')  
file.write('This is line 1.')
file.write('This is line 2.')
print('..Task done')
file.close()

- Here lines will be written one after the other on file because file handling has no nextline character bydefault.
- And default in some version without closing file write action will not get reflected on file.

In [ ]:
file = open(WORK / "msg1.txt", 'wt')
print(file.name)
print(file.mode)
file.write('This is line 1.')
file.write('\n')
file.write('This is line 2.')
file.write('\n')
print('..Task done')
print(file.closed)
file.close()
print(file.closed)

### Use with block
- In writing mode it is important to close file.
- To avoid any error we can use 'with' block then we do not need to close the file explicitely.

#### a mode:
- To avoid overwriting on previously written data, we use append mode.
- `a` or `at` mode is used for writing on file or on text format file respectively in append mode.
- In both 'a' and 'at' mode 
    - if file doesn't exist already then first it will create file then open it.
    - But if file exist then the new content will be written after the previously written data.

In [ ]:
with open(WORK / "msg1.txt", 'at') as file:
    file.write('This is line 3.')
    file.write('\n')
    file.write('This is line 4.')
    file.write('\n')
print('..done')
print(file.closed)

#### x mode:
- `x` or `xt` mode is used for writing on file or on text format file respectively in absolute mode.
- In both 'x' and 'xt' mode 
    - if file doesn't exist already then first it will create file then open it.
    - But if file exist will raise FileExistsError.

In [ ]:
from pathlib import Path

target = WORK / "msg1.txt"

# 'x' is EXCLUSIVE creation - it refuses if the file already exists.
# That is the whole point: it prevents you silently clobbering data.
try:
    with open(target, "xt", encoding="utf-8") as file:
        file.write("This is line 5.\n")
    print("created", target.name)
except FileExistsError as exc:
    print("refused:", exc.strerror, "->", target.name)
    print("  ('w' would have overwritten it without asking)")

In [ ]:
from pathlib import Path

fresh = WORK / "msg_x_demo.txt"
fresh.unlink(missing_ok=True)          # make the cell re-runnable

# First time: succeeds
with open(fresh, "xt", encoding="utf-8") as file:
    file.write("This is line 5.\n")
    file.write("This is line 6.\n")
print("first run :", fresh.name, "created")

# Second time: refuses
try:
    with open(fresh, "xt", encoding="utf-8") as file:
        file.write("overwritten")
except FileExistsError:
    print("second run: refused - the file already exists")

fresh.unlink()          # tidy up
print("cleaned up")

### WAP to create a log file and write user name, age and time of login.

In [ ]:
from time import localtime
def data_entry(fname,uname,uage):
    with open(WORK / fname, 'a') as file:
        file.write(f'Name:{uname}')
        file.write('\n')
        file.write(f'Age:{uage}')
        file.write('\n')
        t = localtime()
        file.write(f'Date:{t[2]}/{t[1]}/{t[0]} {t[3]}:{t[4]}:{t[5]}')
        file.write('\n')
        file.write('--------------------')
        file.write('\n')

In [ ]:
file_name= input('Enter file name with extension: ')
name = input('Enter name: ')
age = int(input('Enter age: '))
data_entry(file_name,name,age)

### Read Methods:
- Read the file: In this step you read the contents of the file. It can be done in several ways:
    - `<file>.read()`
    - `<file>.readline()`
    - `<file>.readlines()`


#### read(value):
- It returns the entire content of the file as a single string.
- Value is the number of bytes to read. In case no value is given then cursor will reads till end the of file.

In [ ]:
file = open(WORK / "msg1.txt", 'rt')
print(file.read()) #Here cursor move character-by-character and reach end of file
file.close()

In [ ]:
file = open(WORK / "msg1.txt", 'rt')
print(file.read())
print(file.read()) # Since cursor reached to end hence nothing prints
file.close()

In [ ]:
file = open(WORK / "msg1.txt", 'rt')
print(file.tell()) # tells current position of cursor
print(file.read())
file.seek(0) # move cursor from current position to required position(index)
print(file.read()) # Since cursor reached to end hence nothing prints
file.close()

In [ ]:
with open(WORK / "msg1.txt", 'rt') as file:
    data = file.read()  # save read string then save in 'data'
print(data)

In [ ]:
with open(WORK / "msg1.txt", 'rt') as file:
    data1 = file.read(4)  # read 4 characters then save in 'data1'
    data2= file.read(4) # read 4 character from cursor current position
    
print(data1)
print(data2)

### WAP to calculate no. of character and word in file:

In [ ]:
# NOTE: the original line was  nwords = len(word)
# `word` was never defined - the cell raised NameError and could never run.
# The variable is `words`.

with open(WORK / "msg1.txt", 'rt', encoding='utf-8') as file:
    data = file.read()

nchar = len(data)
words = data.split()
nwords = len(words)
nlines = len(data.splitlines())

print('File text:')
print(data)
print('No. of lines     :', nlines)
print('No. of words     :', nwords)
print('No. of characters:', nchar)

# ⚠️ .split() with no argument splits on runs of whitespace and drops empties,
# which is what you want for word counting. .split(' ') does not (see 2.1).


### WAP to get the odd Unicode characters read from a file.

In [ ]:
with open(WORK / "msg1.txt", 'rt') as file:
    data = file.read()
    
for ch in data:
    if ord(ch)%2:
        print(f'{ch}: {ord(ch)}', end=' ')

#### readline(value):
- This operation will read a file line-by-line.
- It returns the next line of the file, returning the text up to and including the next newline character.

In [ ]:
with open(WORK / "msg1.txt", 'rt') as file:
    data1 = file.readline()  # read first line including newline then save in 'data1'
    data2= file.read(4) # read 4 character from cursor current position
    
print(data1)
print(data2)

#### readlines():
- It returns a list of the lines in the file, where each item of the list represents a single line.

In [ ]:
with open(WORK / "msg1.txt", 'rt') as file:
    data = file.readlines()
    
print(data)

### Read a particular line from file:

In [ ]:
with open(WORK / "msg1.txt", 'rt') as file:
    data = file.readlines()
    
for line in data:
    if '4' in line:
        print(line)

---

### `pathlib` for file I/O

**7.1** introduced `pathlib` for building and inspecting paths. It also does the reading and
writing, which collapses the most common case to a single line.

| Task | `open()` | `pathlib` |
|---|---|---|
| Read a whole text file | `with open(p) as f: f.read()` | `Path(p).read_text()` |
| Write a whole text file | `with open(p, "w") as f: f.write(s)` | `Path(p).write_text(s)` |
| Read bytes | `with open(p, "rb") as f: f.read()` | `Path(p).read_bytes()` |
| Write bytes | `with open(p, "wb") as f: f.write(b)` | `Path(p).write_bytes(b)` |
| Anything else | `open(p, mode)` | `Path(p).open(mode)` |

**Use `read_text`/`write_text` when the whole file fits in memory and you do it in one go.**
Use `open()` (or `Path.open()`) when you need to append, stream line by line, seek, or hold
the file open across several operations.

⚠️ `write_text` **overwrites**. There is no `append_text` — use `Path.open("at")`.

> Pass `encoding="utf-8"` to these too. They have the same platform-dependent default as
> `open()`.

In [ ]:
from pathlib import Path

demo = WORK / "pathlib_demo.txt"

# ---- write_text / read_text: open, write, close in one call ----
demo.write_text("first line\nsecond line\n", encoding="utf-8")
print("read_text:", repr(demo.read_text(encoding="utf-8")))

# ---- Appending still needs open(), because there is no append_text() ----
with demo.open("at", encoding="utf-8") as handle:
    handle.write("third line\n")

print("after append:", demo.read_text(encoding="utf-8").splitlines())

# ---- Path.open() is just open(), as a method ----
with demo.open(encoding="utf-8") as handle:
    for n, line in enumerate(handle, start=1):
        print(f"  {n}: {line.rstrip()}")

# ---- Binary variants ----
demo.write_bytes(b"\x00\x01\x02\x03")
print("\nread_bytes:", demo.read_bytes())

# ---- Metadata without a separate os.stat call ----
demo.write_text("some content\n", encoding="utf-8")
info = demo.stat()
print("\nsize     :", info.st_size, "bytes")
print("suffix   :", demo.suffix)
print("stem     :", demo.stem)
print("parent   :", demo.parent.name)

# ---- Iterating a directory ----
print("\ntext files in File2Save:")
for p in sorted(Path("File2Save").glob("*.txt")):
    print(f"  {p.name:<24} {p.stat().st_size:>6} bytes")

demo.unlink()
print("\ncleaned up:", not demo.exists())

### Writing-Reading mode:
- `w+`|`a+`|`x+`

In [ ]:
with open(WORK / "msg1.txt",'a+') as file:
    file.write('This is line 6.')
    file.write('\n')
    file.write('This is line 7.')
    file.write('\n')
    file.seek(0)
    data = file.read()

print(data)

### Reading-Writing mode: 'r+'

In [ ]:
with open(WORK / "msg2.txt", 'r+') as file:
    file.seek(len(file.read()))
    file.write('This is line 8.')
    file.write('\n')
    file.seek(0)
    data = file.read()

print(data)

### Read and write other than text file:

In [ ]:
# NOTE: the original version of this cell used './Files/Image.jpg'.
# There is no `Files` directory - the images live in `File2Save`. The path
# was wrong, so this cell raised FileNotFoundError.

from pathlib import Path

source = Path("File2Save/Image.jpg")
print("exists:", source.exists(), "|", source.stat().st_size, "bytes")

with open(source, "rb") as file:        # 'rb' = read binary
    data = file.read()

print("type  :", type(data).__name__)
print("first 20 bytes:", data[:20])

# ⚠️ The original printed the WHOLE image to the notebook. For a 100 KB JPEG
# that is tens of thousands of lines of noise, so we print a slice instead.

with open(WORK / "Image_copy.jpg", "wb") as f2:
    f2.write(data)                      # binary mode takes bytes, not str

print("copied:", (WORK / "Image_copy.jpg").stat().st_size, "bytes")

#### use of file handling:

In [ ]:
# Same path correction as the cell above: ./Files/ -> File2Save/
from pathlib import Path

source = Path("File2Save/Image.jpg")
copy = WORK / "Image_copy.jpg"

# ---- A realistic use of binary file handling: copy and verify ----
with open(source, "rb") as file:
    data = file.read()

with open(copy, "wb") as f2:
    f2.write(data)

# Verify the copy is byte-identical
with open(copy, "rb") as file:
    copied = file.read()

print("sizes match :", len(data) == len(copied))
print("bytes match :", data == copied)

# JPEG files start with the magic bytes FF D8 FF
print("\nmagic bytes :", data[:3].hex(" "))
print("is a JPEG   :", data[:3] == b"\xff\xd8\xff")

print("""
Reading a whole file into memory is fine for a 100 KB image and a bad idea
for a 4 GB one. Chunked reading and file hashing are covered in 8.5.
""")

---

### Atomic writes: never corrupt the file you are replacing

Opening a file in `"w"` mode **truncates it to zero bytes immediately** — before you have
written a single byte of the replacement. If the process dies, the disk fills, or an
exception fires halfway through, you are left with a **half-written file and no original**.

For a scratch file that is fine. For configuration, user data, or anything you would be sad
to lose, use the **write-then-rename** pattern:

```
1. Write the new content to a temporary file in the SAME directory
2. flush() and os.fsync() it, so it is really on disk
3. os.replace(tmp, target)          <- atomic on the same filesystem
```

`os.replace()` is atomic: at every instant the target is either the complete old file or the
complete new one. Never a half of either.

> The temp file **must be in the same directory** as the target. `os.replace()` is only
> atomic within one filesystem, and `/tmp` is often a different one.

**Real-world use case:** this is how editors save files, how package managers update
manifests, and how any daemon rewrites its config without risking a corrupt restart.

In [ ]:
import os, tempfile
from pathlib import Path

target = WORK / "settings.conf"
target.write_text("timeout=30\nretries=3\n", encoding="utf-8")
print("original:", repr(target.read_text(encoding="utf-8")))


# ---- ⚠️ The unsafe pattern ----
def unsafe_update(path: Path, new_content: str, fail: bool = False) -> None:
    with open(path, "wt", encoding="utf-8") as handle:   # truncates IMMEDIATELY
        handle.write(new_content[:10])
        if fail:
            raise RuntimeError("crash halfway through the write")
        handle.write(new_content[10:])


try:
    unsafe_update(target, "timeout=60\nretries=5\nverbose=true\n", fail=True)
except RuntimeError as exc:
    print("\ncrashed:", exc)

print("after crash:", repr(target.read_text(encoding="utf-8")))
print("  ^ the original settings are GONE and the new ones are incomplete")


# ---- ✅ The atomic pattern: write elsewhere, then rename ----
target.write_text("timeout=30\nretries=3\n", encoding="utf-8")   # restore


def atomic_write(path: Path, content: str, fail: bool = False) -> None:
    """Write to a temp file in the same directory, then replace the target."""
    fd, tmp_name = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    tmp = Path(tmp_name)
    try:
        with os.fdopen(fd, "wt", encoding="utf-8") as handle:
            handle.write(content[:10])
            if fail:
                raise RuntimeError("crash halfway through the write")
            handle.write(content[10:])
            handle.flush()
            os.fsync(handle.fileno())        # force it to disk
        os.replace(tmp, path)                # ATOMIC on the same filesystem
    except BaseException:
        tmp.unlink(missing_ok=True)          # leave the original untouched
        raise


try:
    atomic_write(target, "timeout=60\nretries=5\nverbose=true\n", fail=True)
except RuntimeError as exc:
    print("\ncrashed:", exc)

print("after crash:", repr(target.read_text(encoding="utf-8")))
print("  ^ the original is intact - the partial write went to a temp file")

atomic_write(target, "timeout=60\nretries=5\nverbose=true\n")
print("\nafter success:", repr(target.read_text(encoding="utf-8")))

target.unlink()

### Writing Output on file using print()
- This means that whatever we try to print will be saved to a file. This can come handy when:
    - We don’t want to convert our project to use logger temporarily
    - Keep the print statements handy and portable
<br/><br/>
- Printing to file: Print our output to files can be achieved in two ways:
    - 1. Setting the route as global
    - 2. Deciding with each print call

In [ ]:
# 🔴 The original version of this cell was:
#
#     import sys
#     sys.stdout = open('File2Save/output1.txt','wt')
#     print("Hello Python!")
#
# It replaced sys.stdout globally and never restored it, so EVERY later
# print in the session went silently into that file - and the file was
# never closed. In a notebook that is close to unrecoverable without a
# kernel restart.
#
# Use contextlib.redirect_stdout instead: it restores automatically, even
# if the block raises.

from contextlib import redirect_stdout
from pathlib import Path

out_path = WORK / "output1.txt"

with open(out_path, "wt", encoding="utf-8") as handle:
    with redirect_stdout(handle):
        print("Hello Python!")
        print("We are printing to file.")

# stdout is back to normal here
print("wrote to", out_path.name)
print("contents:", repr(out_path.read_text(encoding="utf-8")))

In [ ]:
# 🔴 The original version was:
#
#     print("Each statement will print.", file=open('File2Save/output1.txt','a'))
#     print("Next statement will print.", file=open('File2Save/output.txt1','a'))
#
# Two problems:
#   1. Each open() leaks a file handle - nothing ever closes them, so you
#      get a ResourceWarning and the write may not be flushed.
#   2. 'output.txt1' is a typo for 'output1.txt', which is why a stray
#      file with that name appeared in File2Save.

from pathlib import Path

log_path = WORK / "output1.txt"

# Correct: one open handle, closed by `with`, several prints into it
with open(log_path, "at", encoding="utf-8") as handle:
    print("Each statement will print.", file=handle)
    print("Next statement will print.", file=handle)

print("file now contains:")
for line in log_path.read_text(encoding="utf-8").splitlines():
    print("  ", line)

# print()'s `file` argument accepts anything with a .write() method -
# which is why redirect_stdout and StringIO work the same way (5.4).
import io
buffer = io.StringIO()
print("straight into memory", file=buffer)
print("\nStringIO got:", repr(buffer.getvalue()))

---

## Common Mistakes & Pitfalls

1. 🔴 **Omitting `encoding=`.** The default is platform-dependent, so the same code produces different files on Windows and Linux. Always pass `encoding="utf-8"`.
2. 🔴 **Assigning to `sys.stdout` without restoring it.** Every later `print` disappears into the file. Use `contextlib.redirect_stdout`.
3. **Not closing files.** `open()` without `with` leaves the handle open until garbage collection — and buffered writes may never reach disk.
4. **Opening in `"w"` to update a file.** It truncates immediately; a crash loses both the old and the new content. Use the atomic write-then-rename pattern.
5. **`file.read()` on a large file.** It loads the whole thing into memory. Iterate the file object instead.
6. **Reading twice without seeking.** After `read()` the cursor is at the end, so the second `read()` returns `""`.
7. **Forgetting `\n` in `write()`.** Unlike `print()`, `write()` adds nothing.
8. **Using a relative path and assuming the working directory.** Build paths from a known root (**7.1**).
9. **`readlines()` when you only need to iterate.** `for line in file:` is lazy and does the same job.

## Best Practices

- **Always** use `with` — it closes the file even if the block raises.
- **Always** pass `encoding="utf-8"` unless you have a specific reason not to.
- Iterate the file object (`for line in f:`) rather than `readlines()`.
- Use `Path.read_text()` / `write_text()` for whole small files.
- Use `x` mode when creating a file that must not already exist.
- Use the atomic write pattern for anything you would be sad to lose.
- Strip line endings with `.rstrip("\n")` rather than `.strip()`, which also eats meaningful leading whitespace.
- Use `utf-8-sig` when reading files produced by Excel.

## Practice Exercises

Try these before moving on.

1. Write a file with non-ASCII text, then read it with the wrong encoding. Do it once so it raises and once so it silently corrupts.
2. Count the words, lines and characters in a file, reading it only once.
3. Copy a file line by line without ever loading more than one line into memory.
4. Implement `atomic_write()` and prove the original survives a mid-write exception.
5. Read the last 5 lines of a file efficiently. What is the naive approach's problem?
6. Use `seek`/`tell` to read a file, note a position, read on, then return to it.
7. Write a function that reads a file and returns `None` instead of raising if it is missing.
8. Redirect `print()` to a file for one block, and confirm normal output resumes afterwards.

In [ ]:
# Remove the scratch directory. Nothing this notebook wrote ever touched
# the repository, so re-running it from the top always gives the same result.
import shutil

shutil.rmtree(WORK, ignore_errors=True)
print("cleaned up:", WORK)
print("still exists:", WORK.exists())